In [ ]:
# pip install pykalman
# %pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np

ramdom_state = 42
# df = pd.read_csv('data/relax_class.csv')
df = pd.read_csv('C:/Users/hiros/OneDrive - 学校法人立命館/デスクトップ/Lab/研究/実装/TICC/data/neck_classed.csv')
df

In [ ]:
# パラメータ指定
method = 'poly'  # 'poly' or 'ar' or 'kalman'
degree_or_lags = 3  # 多項式次数 or ARラグ数
n_estimators = 4  # 決定木の本数
segment_length = 10  # セグメントの長さ
step = 1  # セグメントのステップ幅

In [ ]:
df.columns

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np
from numpy.polynomial.polynomial import polyfit
from statsmodels.tsa.ar_model import AutoReg
from pykalman import KalmanFilter

def kalman_filter_series(series):
    # カルマンフィルタの初期化（状態は観測と同次元で、初期値は最初の観測値）
    kf = KalmanFilter(initial_state_mean=series.iloc[0], n_dim_obs=1)

    # フィルタ処理を実行：ノイズを考慮しながら系列の滑らかな推定値を返す
    state_means, _ = kf.filter(series.values.reshape(-1, 1))

    # フィルタ結果を pd.Series に変換
    return pd.Series(state_means.flatten(), index=series.index)



# 特徴抽出関数：多項式近似 or ARモデル or カルマンフィルタを用いて、各セグメントの特徴量を抽出する
def vectorize_sequence(df, method='poly', degree_or_lags=3):
    features = {}
    t = df['time'].values
    
    # もし、カラムにclassがあれば削除,なければそのまま
    if 'class' in df.columns:
        cols = df.columns.drop(['time','class'])
    else:
        cols = df.columns.drop(['time'])

    cols = ['acc_x','acc_y','acc_z','ax_f','ay_f','az_f','vx','vy','vz']

    for col in cols:
        y = df[col].values

        if method == 'poly':
            try:
                coefs = polyfit(t, y, degree_or_lags)
            except:
                coefs = np.zeros(degree_or_lags + 1)
            for i, c in enumerate(coefs):
                features[f'{col}_coef{i}'] = c

        elif method == 'ar':
            try:
                model = AutoReg(y, lags=degree_or_lags, old_names=False).fit()
                for i, coef in enumerate(model.params):
                    features[f'{col}_ar{i}'] = coef
            except:
                for i in range(degree_or_lags + 1):
                    features[f'{col}_ar{i}'] = 0.0

        elif method == 'kalman':
            try:
                y_kf = kalman_filter_series(df[col])
                features[f'{col}_kf_mean'] = y_kf.mean()
                features[f'{col}_kf_std'] = y_kf.std()
                features[f'{col}_kf_min'] = y_kf.min()
                features[f'{col}_kf_max'] = y_kf.max()

                try:
                    ac = y_kf.autocorr(lag=1)
                    features[f'{col}_kf_autocorr'] = ac if not np.isnan(ac) else 0.0
                except:
                    features[f'{col}_kf_autocorr'] = 0.0
            except:
                features[f'{col}_mean'] = 0.0
                features[f'{col}_std'] = 0.0
                features[f'{col}_min'] = 0.0
                features[f'{col}_max'] = 0.0

    return features


def train_binary_trees_by_class(df_all, method='poly', degree_or_lags=3,
                                 segment_length=50, step=50):
    """
    各クラスに対して、1本の決定木で「そのクラス vs その他」の2値分類器を学習する。

    Returns:
    - models: dict {クラス番号: 学習済みモデル}
    - feature_names: 特徴量名一覧（あとで予測時に必要）
    """

    df_all = df_all[df_all['class'].notnull()].copy()
    df_all['class'] = df_all['class'].astype(int)
    classes = sorted(df_all['class'].unique())

    models = {}
    for target_class in classes:
        feature_list = []
        label_list = []

        for label in classes:
            df_label = df_all[df_all['class'] == label]
            for start in range(0, len(df_label) - segment_length + 1, step):
                segment = df_label.iloc[start:start + segment_length]
                vec = vectorize_sequence(segment, method=method, degree_or_lags=degree_or_lags)
                # バイナリラベル化：target_classなら1、それ以外は0
                vec['label'] = 1 if label == target_class else 0
                feature_list.append(vec)

        df_vec = pd.DataFrame(feature_list)
        X = df_vec.drop(columns='label')
        y = df_vec['label'].astype(int)

        # 決定木（深さ制限などは好みに応じて追加）
        model = DecisionTreeClassifier(random_state=42,criterion='entropy')
        model.fit(X, y)

        models[target_class] = model
        feature_names = X.columns.tolist()  # 最後に更新されたものでOK（すべて同じはず）

    return models, X,y,feature_names


In [ ]:
# モデルの学習結果を確認
model, X, y, feature_names = train_binary_trees_by_class(df, method=method, degree_or_lags=degree_or_lags,
                                                        segment_length=segment_length, step=step)
print("学習済みモデル:", model)
print("特徴量名:", feature_names)
print("特徴量のサンプル:\n", X.head())
print("ラベルのサンプル:\n", y.head())

In [ ]:
X.head()

In [ ]:
def fill_short_unclassified(pred_labels, max_fill_length=2):
    """
    -1（未分類）が連続して max_fill_length 以下の長さなら、
    直前のクラスで埋める。長ければそのまま。
    """
    pred_labels = list(pred_labels)  # 可変にする
    i = 0
    while i < len(pred_labels):
        if pred_labels[i] == -1:
            # -1の開始位置を記録
            start = i
            while i < len(pred_labels) and pred_labels[i] == -1:
                i += 1
            end = i
            length = end - start
            # 補完条件チェック
            if length <= max_fill_length and start > 0:
                fill_value = pred_labels[start - 1]
                for j in range(start, end):
                    pred_labels[j] = fill_value
        else:
            i += 1
    return pred_labels


def predict_with_binary_trees(models, df_predict, feature_names,
                              method='poly', degree_or_lags=3,
                              segment_length=50, step=50):
    """
    各セグメントに対して、全クラスの2値分類器（各クラスに1本の木）で予測。
    出力は 0/1 のベクトル [0, 1, 0, 0] のような形式。

    - 複数の1が出た場合 → クラス番号の小さい方を選ぶ（または他のルールでもOK）
    - 全部0の場合 → -1（未分類ラベル）

    Returns:
    - pred_labels: 最終的なクラス予測（または -1）
    - bin_outputs: 各セグメントの [0/1, 0/1, ..., 0/1] 出力
    """

    feature_list = []
    segment_starts = []

    for start in range(0, len(df_predict) - segment_length + 1, step):
        segment = df_predict.iloc[start:start + segment_length]
        vec = vectorize_sequence(segment, method=method, degree_or_lags=degree_or_lags)
        feature_list.append(vec)
        segment_starts.append(start)

    df_vec = pd.DataFrame(feature_list)[feature_names]

    bin_outputs = []
    pred_labels = []
    all_probas = []

    for i in range(len(df_vec)):
        row = df_vec.iloc[i:i+1]
        votes = []
        probas = []

        # 各クラス用モデルに通して 0/1 の結果を得る
        for cls, model in models.items():
            pred = model.predict(row)[0]  # 単一予測（0 または 1）
            votes.append(pred)
            proba = model.predict_proba(row)[0][1]
            probas.append(proba)

        bin_outputs.append(votes)
        all_probas.append(probas)

        # 予測ルール：
        # - votes = [0, 1, 0, 0] → 1
        # - votes = [0, 1, 1, 0] → 最初の 1 の index（今回は 1）
        # - votes = [0, 0, 0, 0] → 未分類 = -1
        if sum(votes) == 0:
            pred_labels.append(-1)  # どのクラスにも属さない
        else:
            pred_class = np.where(np.array(votes) == 1)[0][0]  # 最初の 1 の位置（クラス番号）
            pred_labels.append(pred_class)

    # 補完処理：短い未分類セグメントを直前のクラスで埋める
    pred_labels = fill_short_unclassified(pred_labels)
    
    return pred_labels, bin_outputs,all_probas,df_vec,segment_starts


In [ ]:
start = 0

df_test = df.copy()

df_test_noisy = df_test.copy()
num_cols = df_test_noisy.select_dtypes(include=[np.number]).columns

# 正規分布でノイズを生成して加算
noise = np.random.normal(loc=0, scale=.2, size=df_test[num_cols].shape)
df_test_noisy[num_cols] = df_test[num_cols] + noise

df_test = df_test_noisy.drop(columns='class')

df_test = df_test.iloc[start:100]

In [ ]:
# 予測データ: df_test（sequence_counter, acc_x などを含む）
pred_labels,bin_output, all_probs, df_vec,segment_starts = predict_with_binary_trees(
    models=model,
    df_predict=df_test,
    feature_names=feature_names,
    method=method,
    degree_or_lags=degree_or_lags,
    segment_length=segment_length,
    step=step
)

print("予測ラベルのサンプル:", pred_labels[:10])
print("予測確率のサンプル:", all_probs[:10])
print("セグメント開始インデックスのサンプル:", segment_starts[:10])
print("バイナリ出力のサンプル:", bin_output[:10])
# 予測結果をデータフレームにまとめる
df_results = pd.DataFrame({
    'segment_start': segment_starts,
    'predicted_class': pred_labels,
    'binary_output': bin_output,
    'probabilities': all_probs
})
df_results

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 正解ラベル（セグメント単位）を作成
test_labels = df['class'].fillna(-1).astype(int).reset_index(drop=True)
segment_labels = []
for start in start + np.arange(0, len(df_test) - segment_length + 1, step):
    segment = test_labels.iloc[start:start + segment_length]
    valid = segment[segment != -1]
    label = valid.mode().iloc[0] if not valid.empty else -1
    segment_labels.append(label)

segment_labels = np.array(segment_labels)
pred_labels = np.array(pred_labels)

true_valid = segment_labels
pred_valid = pred_labels

print("正解ラベルのサンプル:", true_valid)
print("予測ラベルのサンプル:", pred_valid)

# 1. Accuracy
print("Accuracy:", accuracy_score(true_valid, pred_valid))



In [ ]:
df_score = pd.DataFrame({
    'true': true_valid,
    'pred': pred_valid
})
df_score['correct'] = df_score['true'] == df_score['pred']
df_score

In [ ]:
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

methods = ['poly', 'ar', 'kalman']
results = []

for method in methods:
    print(f"\n--- Method: {method} ---")
    
    # 学習（各クラスごとの2値分類器を構築）
    models, X,y,feature_names = train_binary_trees_by_class(df, method=method,degree_or_lags=degree_or_lags,segment_length=segment_length, step=step)

    # 予測
    pred_labels, bin_output, all_probs, df_vec, segment_starts = predict_with_binary_trees(
        models=models,
        df_predict=df_test,
        feature_names=feature_names,
        method=method,
        degree_or_lags=degree_or_lags,
        segment_length=segment_length,
        step=step
    )

    # 正解ラベル作成（多数決）
    test_labels = df['class'].fillna(-1).astype(int).reset_index(drop=True)
    segment_labels = []
    for start in segment_starts:
        segment = test_labels.iloc[start:start + segment_length]
        valid = segment[segment != -1]
        label = valid.mode().iloc[0] if not valid.empty else -1
        segment_labels.append(label)

    segment_labels = np.array(segment_labels)
    pred_labels = np.array(pred_labels)

    # # マスク（-1 を除外）
    # mask = (segment_labels != -1) & (pred_labels != -1)
    # pred_valid = pred_labels[mask]
    # true_valid = segment_labels[mask]
    true_valid = segment_labels
    pred_valid = pred_labels

    acc = accuracy_score(true_valid, pred_valid)
    dim = len(feature_names)

    results.append({
        'method': method,
        'accuracy': acc,
        'dimension': dim
    })
    print(f"Accuracy: {acc:.4f}, Dimension: {dim}")


In [ ]:
print(true_valid)
print(pred_valid)


In [ ]:
df_result = pd.DataFrame(results)

# 精度の可視化
plt.figure(figsize=(10, 4))
sns.barplot(x='method', y='accuracy', data=df_result)
plt.title('Accuracy by Feature Extraction Method')
plt.ylim(0, 1)
plt.show()

# 特徴量次元数の可視化
plt.figure(figsize=(10, 4))
sns.barplot(x='method', y='dimension', data=df_result)
plt.title('Feature Dimension by Method')
plt.show()


In [ ]:
from sklearn.tree import plot_tree
# 決定木の可視化
plt.figure(figsize=(20, 10))
plot_tree(model[0], feature_names=feature_names, filled=True, fontsize=10)
plt.title('Decision Tree for Class 0 vs Others')
plt.show()


今、決定木1本あたり１クラスを見つけている

決定木１本で多クラス分類をする。決定木１本を一つの行動に充てる
行動内の全クラスを見つける\
plot_treeで可視化もする